# Fast No RL 03 - Evaluation-Weight Tuning and Development Games

Play the selected classical or supervised variant through the preserved process harness on CPU. The classical path requires no checkpoint. It tunes numeric evaluation coefficients through matches, not RL or neural training. Actual PGNs, move clocks, logs, and CSV scores are saved.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project Setup

Uses `configs/fast_no_rl.yaml`. No league, self-play, PPO, or DQN is run. Colab's installed PyTorch is preserved.

In [ ]:
from pathlib import Path
import sys
import subprocess
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def show_figure(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl/non_rl.py").is_file():
    raise FileNotFoundError(f"Place the updated project contents directly in {PROJECT_ROOT}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT_ROOT / "requirements_colab.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
from chess_rl.non_rl import load_no_rl_config
from chess_rl.reproducibility import read_json, sha256
cfg = load_no_rl_config(PROJECT_ROOT, "fast_no_rl.yaml")
print("Run:", cfg["run_id"], "| Variant:", cfg["non_rl"]["variant"])
print("Root:", PROJECT_ROOT)


## Inspect the Match Settings

Defaults are 256 games per opponent, with colour-swapped opening pairs, at 120 seconds + 0.5 seconds. Changing parameters is an experiment: use a new run_id. Game results depend on the local CPU, not just the GPU allocation.

In [ ]:
print(cfg["evaluation"])

## Tune Classical Evaluation Weights

Default: compare eight seeded coefficient sets against the fixed starting engine, with 32 games per trial. Confirm only the best eligible trial over 128 games on unused development openings. Promote only with score >= 55%, an opening-family bootstrap lower bound above 50%, and no recorded runtime failures on either side. Otherwise keep the baseline. These defaults are experiment settings, not established optimal values. Held-out openings are never used to select weights. Edit `non_rl.tuning.ranges` or `search.evaluation_weights` in the config before starting a new run. Set `non_rl.tuning.enabled: false` to evaluate your manually chosen weights directly. Supervised runs skip this classical tuner.

In [ ]:
from chess_rl.evaluation_tuning import tune_classical_weights
tuning = tune_classical_weights(PROJECT_ROOT, cfg)
if tuning is not None:
    print("Decision:", tuning["decision"])
    print("Selected coefficients:", tuning["selected_weights"])
    print("Trial CSV:", PROJECT_ROOT / "results" / cfg["run_id"] / "no_rl/weight_tuning/trials.csv")
else:
    print("Classical weight tuning disabled or supervised variant selected.")

## Inspect Screening Results

Screening scores select a candidate; they are not independent evidence of improvement. The separate confirmation match decides whether its weights are kept. Notebook 04 subsequently measures the frozen choice on held-out games.

In [ ]:
if tuning is not None:
    from chess_rl.plots import plot_matches
    trial_summaries = {f'Trial {row["trial"]}': row["summary"] for row in tuning["trials"]}
    if any(row["score"] is not None for row in trial_summaries.values()):
        show_figure(plot_matches(PROJECT_ROOT, cfg["run_id"], trial_summaries, "no_rl_weight_screening"))
    else:
        print("No scored screening games. Inspect the saved failure logs.")
    print("Confirmation:", tuning["confirmation"] or "Not run: no usable screening score above 50%.")

## Run or Resume Development Games

Opponents: greedy, minimax, and the preserved original classical agent. For a supervised run, also evaluate a matched no-network control and play the supervised agent against that control. Completed games are reused only when their inputs match.

In [ ]:
from chess_rl.non_rl import evaluate_no_rl
development = evaluate_no_rl(PROJECT_ROOT, cfg)
print("Saved results:", PROJECT_ROOT / "results" / cfg["run_id"] / "no_rl/development.json")

## Compare Scores

Draws contribute half a point. Intervals use opening-family paired bootstrap; they are unavailable when too few independent pairs were played. Do not claim an improvement from an uncertain interval.

In [ ]:
from chess_rl.plots import plot_matches
show_figure(plot_matches(PROJECT_ROOT, cfg["run_id"], development["summaries"], "no_rl_development"))
if development["classical_control"]:
    show_figure(plot_matches(PROJECT_ROOT, cfg["run_id"], development["classical_control"], "no_rl_classical_control"))

## Find the Games

Open PGNs using a chess viewer to replay full games. The saved files below are from this run only.

In [ ]:
match_root = PROJECT_ROOT / "results" / cfg["run_id"] / "matches"
for pgn in sorted(match_root.glob("no-rl-dev-*/game-*.pgn"))[:5]:
    print(pgn)
print("Next: no-RL notebook 04 freezes this variant before held-out evaluation.")